# House Prices — Step 5: K-Fold Target Encoding

Plain Target Encoding (Step 4) computes each category's mean `SalePrice` using **all** rows — including the very row being encoded. That's target leakage: the row's own label partially determines its own feature value, which inflates a model's apparent performance during training and can hurt generalization.

**K-Fold Target Encoding** removes this leakage:

1. Split the training data into K folds (e.g. K=5).
2. For each fold, compute category means using **only the other K-1 folds**.
3. Use those out-of-fold means to encode the held-out fold.
4. Repeat until every row has been encoded using data it wasn't part of.

This is the version you'd actually want to use as a model input feature. We still apply smoothing (as in Step 4) to protect against rare categories within each fold's training data.

## 1. Load the cleaned dataset

Continuing from `train_cleaned.csv` (Step 1).

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold

pd.set_option('display.max_columns', 15)

df = pd.read_csv('train_cleaned.csv')
print("Shape:", df.shape)
df[['Neighborhood', 'SalePrice']].head(5)

Shape: (1460, 81)


,Neighborhood,SalePrice
0,CollgCr,208500
1,Veenker,181500
2,CollgCr,223500
3,Crawfor,140000
4,NoRidge,250000


## 2. See the fold logic in action, on `Neighborhood`

Let's split the data into 5 folds and, for just the first fold, compute `Neighborhood` means using **only the other 4 folds** — the same logic we'll apply to every column next.

In [2]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
folds = list(kf.split(df))

train_idx, val_idx = folds[0]
train_fold = df.iloc[train_idx]
val_fold = df.iloc[val_idx]

fold_means = train_fold.groupby('Neighborhood')['SalePrice'].mean()
print(f"Fold 1 validation rows: {len(val_fold)}, encoded using means from the other {len(train_fold)} rows")
val_fold[['Neighborhood']].assign(
    Neighborhood_encoded=val_fold['Neighborhood'].map(fold_means)
).head(8)

Fold 1 validation rows: 292, encoded using means from the other 1168 rows


,Neighborhood,Neighborhood_encoded
15,BrkSide,122640.000000
23,MeadowV,101890.000000
29,BrkSide,122640.000000
30,IDOTRR,106623.076923
32,CollgCr,201112.008696
43,CollgCr,201112.008696
44,NAmes,147408.204420
49,Sawyer,136867.810345


## 3. A reusable K-Fold Target Encoding function

This function combines the fold logic above with the smoothing formula from Step 4, and handles a category that appears in the validation fold but not in that fold's training data (falls back to the global mean).

In [3]:
def kfold_target_encode(df, col, target_col, n_splits=5, m=10, random_state=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    global_mean = df[target_col].mean()
    encoded = pd.Series(index=df.index, dtype=float)

    for train_idx, val_idx in kf.split(df):
        train_fold = df.iloc[train_idx]
        agg = train_fold.groupby(col)[target_col].agg(['mean', 'count'])
        smoothed = (agg['count'] * agg['mean'] + m * global_mean) / (agg['count'] + m)

        mapped = df.iloc[val_idx][col].map(smoothed)
        encoded.iloc[val_idx] = mapped.values

    encoded = encoded.fillna(global_mean)  # category unseen in that fold's training data
    return encoded

# quick test on Neighborhood
encoded_neighborhood = kfold_target_encode(df, 'Neighborhood', 'SalePrice')
df[['Neighborhood', 'SalePrice']].assign(Neighborhood_kfold_encoded=encoded_neighborhood).head(8)

,Neighborhood,SalePrice,Neighborhood_kfold_encoded
0,CollgCr,208500,196200.387278
1,Veenker,181500,197950.747432
2,CollgCr,223500,196200.387278
3,Crawfor,140000,209292.570590
4,NoRidge,250000,307844.650207
5,Mitchel,143000,164164.850189
6,Somerst,307000,220691.399452
7,NWAmes,200000,186994.367043


## 4. Apply K-Fold Target Encoding to all categorical columns

We repeat this for every categorical column, producing the leak-free version of the dataset.

In [4]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Encoding {len(cat_cols)} categorical columns")

df_kfold_encoded = df.copy()

for col in cat_cols:
    df_kfold_encoded[col] = kfold_target_encode(df, col, 'SalePrice', n_splits=5, m=10)

print("Shape unchanged:", df_kfold_encoded.shape)
df_kfold_encoded.head(3)

Encoding 43 categorical columns


/tmp/ipykernel_89/936946745.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


Shape unchanged: (1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,...,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,191233.328298,65.0,8450,181302.514031,180921.19589,...,180921.19589,0,2,2008,173064.519491,175105.214094,208500
1,2,20,191788.168068,80.0,9600,181345.610366,180921.19589,...,180921.19589,0,5,2007,172962.063810,174628.848879,181500
2,3,60,191233.328298,68.0,11250,181302.514031,180921.19589,...,180921.19589,0,9,2008,173064.519491,175105.214094,223500


## 5. Compare against the plain (leaky) Target Encoding

A quick sanity check: the K-Fold encoded values should be close to, but not identical to, the plain target-encoded values from Step 4 — the difference reflects the leakage we removed.

In [5]:
df_plain = pd.read_csv('train_target_encoded.csv')

comparison = pd.DataFrame({
    'Neighborhood_plain': df_plain['Neighborhood'],
    'Neighborhood_kfold': df_kfold_encoded['Neighborhood']
})
print("Correlation between plain and K-Fold encodings:", comparison.corr().iloc[0, 1].round(4))
comparison.head(8)

Correlation between plain and K-Fold encodings: 0.9977


,Neighborhood_plain,Neighborhood_kfold
0,196900.487243,196200.387278
1,211224.378995,197950.747432
2,196900.487243,196200.387278
3,205755.294408,209292.570590
4,305025.881547,307844.650207
5,160448.270490,164164.850189
6,220748.728739,220691.399452
7,188070.686252,186994.367043


## 6. Save the K-Fold target-encoded dataset

This is the version to use as model input for training. A quick note for later: when encoding the **test set**, you don't repeat the K-Fold process — instead, you use the category means computed from the **entire training set** (the smoothed version from Step 4), since there's no leakage risk when the statistic comes from training data and is applied to separate test rows.

In [6]:
df_kfold_encoded.to_csv('train_kfold_target_encoded.csv', index=False)
print("Saved train_kfold_target_encoded.csv with shape:", df_kfold_encoded.shape)

Saved train_kfold_target_encoded.csv with shape: (1460, 81)
